# 🎙️ Narraciones con tu voz · Guía Clase N°1 · M1

Este cuaderno genera las narraciones de los 40 videos **con tu voz**, usando el modelo gratuito y de código abierto *Chatterbox Multilingual* (licencia MIT).

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar**.

Luego: **Entorno de ejecución → Ejecutar todo**. Tu voz ya está cargada (paso 3). El cuaderno te dejará escuchar una prueba (paso 5) y al final descargará `audios.zip`.

## 1. Revisar que haya GPU

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("❌ No hay GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4, y vuelve a ejecutar todo.")
print("✅ GPU lista:", r.stdout.strip())

## 2. Instalar el modelo (tarda 2–4 minutos)
La primera vez, al terminar la instalación **la sesión se reinicia sola** (es normal). Entonces vuelve a usar **Entorno de ejecución → Ejecutar todo**: la segunda vez la instalación se salta y sigue directo.

In [ ]:
import importlib.util, os
if importlib.util.find_spec("chatterbox") is None:
    get_ipython().system("pip install -q chatterbox-tts")
    print("✅ Instalado. La sesión se reiniciará sola para aplicar los cambios.")
    print("👉 Cuando aparezca el aviso de reinicio, vuelve a: Entorno de ejecución → Ejecutar todo.")
    os.kill(os.getpid(), 9)
else:
    print("✅ El modelo ya está instalado")

## 3. Tu voz de referencia
Ya está cargada automáticamente desde tu repositorio (tomada de tu video). **No necesitas subir nada.** Si algún día quieres usar otra grabación, cambia `SUBIR_OTRA = False` por `True`.

In [ ]:
import os, subprocess, urllib.request
SUBIR_OTRA = False
if SUBIR_OTRA:
    from google.colab import files
    subido = files.upload(); nombre = list(subido.keys())[0]
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",nombre,"-af","silenceremove=start_periods=1:start_threshold=-45dB",
                    "-ac","1","-ar","24000","-t","25","voz_ref.wav"], check=True)
elif not os.path.exists("voz_ref.wav"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/profcristobalespinoza-netizen/m1-clase-1/main/voz/voz_ref.wav", "voz_ref.wav")
print("✅ Audio de referencia listo")
from IPython.display import Audio, display
display(Audio("voz_ref.wav"))

## 4. Cargar el modelo y los guiones

In [ ]:
import json, urllib.request, torch, torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
try:
    model = ChatterboxMultilingualTTS.from_pretrained(device="cuda", t3_model="v3")
except TypeError:
    model = ChatterboxMultilingualTTS.from_pretrained(device="cuda")
GUIONES = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/profcristobalespinoza-netizen/m1-clase-1/main/voz/guiones.json"))
print("✅ Modelo cargado ·", len(GUIONES), "ejercicios")

import re
def frases(texto):
    partes = re.split(r'(?<=[.!?¿¡:])\s+', texto.strip())
    out, buf = [], ""
    for p in partes:
        if len(buf) + len(p) < 220: buf = (buf + " " + p).strip()
        else:
            if buf: out.append(buf)
            buf = p
    if buf: out.append(buf)
    return out

def narrar(texto, destino):
    trozos = []
    pausa = torch.zeros(1, int(model.sr * 0.30))
    for f in frases(texto):
        wav = model.generate(f, language_id="es", audio_prompt_path="voz_ref.wav")
        trozos += [wav.cpu(), pausa]
    ta.save(destino, torch.cat(trozos, dim=1), model.sr)

## 5. Prueba: escucha cómo suena
Si no te convence, sube otro audio de referencia (borra `voz_ref.wav` en la carpeta de la izquierda y vuelve a ejecutar el paso 3).

In [ ]:
narrar(GUIONES["1"][0], "prueba.wav")
display(Audio("prueba.wav"))

## 6. Generar los 40 ejercicios (30–60 minutos)
No cierres la pestaña. Si se corta la conexión, vuelve a ejecutar esta celda: continúa donde quedó (mientras la sesión siga activa).

In [ ]:
import os, time
os.makedirs("audios", exist_ok=True)
total = sum(len(v) for v in GUIONES.values()); hechos = 0; t0 = time.time()
for n in sorted(GUIONES, key=int):
    for k, texto in enumerate(GUIONES[n]):
        destino = f"audios/ej{int(n):02d}_{k:02d}.wav"
        if not os.path.exists(destino):
            narrar(texto, destino)
        hechos += 1
    print(f"Ejercicio {n} listo · {hechos}/{total} tramos · {int(time.time()-t0)//60} min")
print("✅ Todas las narraciones generadas")

## 7. Descargar
Se descarga `audios.zip`. Súbelo al chat con Claude para armar los videos.

In [ ]:
!cd audios && zip -q -r ../audios.zip .
from google.colab import files
files.download("audios.zip")